# Modern Coding Practice — Concurrent Web Crawler (Amazon FAR style)

Crawl a link graph from a seed URL. Parts 1-3 build the core (concurrency at Part 3 — including the
classic "how do the workers know when they're *done*?" problem); Parts 4-5 are stretch tasks (robust
crawling, then parallel parsing). The "web" is a `dict[url -> list[url]]` so tests are deterministic.
Fill stubs, run each test cell, peek at the collapsed `ref_` solutions only after trying.

---

## Part 1 — Extract links

`get_links(graph, url) -> list[str]`: the page's outlinks, **de-duplicated (order preserved)** and
**excluding self-links**. An unknown URL (not a key in `graph`) yields `[]`.

**Lock down:** dedupe while keeping first-seen order; drop self-loops; unknown pages are empty, not an
error (error handling is Part 4).

In [ ]:
def get_links(graph, url):
    raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    g = {"a": ["b", "c", "b", "a", "d"]}
    assert get_links(g, "a") == ["b", "c", "d"]     # dedupe, drop self-link "a"
    assert get_links(g, "z") == []                  # unknown page
    print("Part 1 OK")

_t1()

## Part 2 — Breadth-first crawl (single-threaded)

`crawl_bfs(graph, seed, max_depth) -> list[str]`: BFS from `seed` (depth 0), returning URLs in the
order first discovered, never revisiting, and not expanding beyond `max_depth` (a page at `max_depth`
is recorded but its links aren't followed).

**Lock down:** a `visited` set prevents cycles/repeats; BFS uses a FIFO queue; `max_depth=0` returns
just the seed.

In [ ]:
from collections import deque


def crawl_bfs(graph, seed, max_depth):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    g = {"a": ["b", "c"], "b": ["d"], "c": ["d", "e"], "d": [], "e": ["a"]}
    assert crawl_bfs(g, "a", 10) == ["a", "b", "c", "d", "e"]
    assert crawl_bfs(g, "a", 1) == ["a", "b", "c"]      # only the seed's neighbours
    assert crawl_bfs(g, "a", 0) == ["a"]
    print("Part 2 OK")

_t2()

## Part 3 — Concurrent crawl

`concurrent_crawl(graph, seed, num_workers) -> set[str]`: worker threads pull URLs from a shared
frontier, fetch their links, and enqueue unseen ones, with a thread-safe `visited` set, until **all
reachable URLs are crawled**. Return the visited set (every URL reachable from `seed`).

**The crux — termination detection:** workers block on an empty frontier, but "empty right now"
doesn't mean "done." Use `queue.Queue` with `task_done()` + `join()` to know when every enqueued URL
has been processed, then send one sentinel per worker to stop them. **Discuss:** the check-then-add
race on `visited` (guard it), why a bare `get()`-with-timeout is flaky, and how this differs from a
fixed-size producer/consumer.

In [ ]:
import queue
import threading


def concurrent_crawl(graph, seed, num_workers):
    raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    g = {"a": ["b", "c"], "b": ["c", "d"], "c": ["a"], "d": ["e"], "e": [],
         "x": ["y"], "y": []}                            # x, y unreachable from a
    assert concurrent_crawl(g, "a", 8) == {"a", "b", "c", "d", "e"}
    big = {i: [(i * 2 + 1) % 50, (i * 2 + 2) % 50] for i in range(50)}
    expected = set(crawl_bfs(big, 0, 10 ** 9))
    for _ in range(5):                                   # repeat to surface races
        assert concurrent_crawl(big, 0, 8) == expected
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Robust crawl

Real pages have dead links, cycles, and depth budgets. `robust_crawl(graph, seed, max_depth) ->
(visited, dead_count)`:
- skip **dead links** (targets that aren't keys in `graph`) without crashing, counting **distinct**
  dead targets;
- never revisit (cycles are fine);
- respect `max_depth`;
- if the seed itself is unknown, return `(set(), 0)`.

**Discuss:** where dead-link / retry / robots / canonicalization policy belongs, and how you'd add
bounded retries for transient fetch failures.

In [ ]:
def robust_crawl(graph, seed, max_depth):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    g = {"a": ["b", "x"], "b": ["c", "x"], "c": []}      # x is a dead link (not a key)
    visited, dead = robust_crawl(g, "a", 10)
    assert visited == {"a", "b", "c"} and dead == 1      # x counted once across both pages
    v2, _ = robust_crawl(g, "a", 1)
    assert v2 == {"a", "b"}                              # depth-limited
    assert robust_crawl({"a": []}, "zzz", 5) == (set(), 0)   # unknown seed
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel fetch/parse

`parallel_fetch_parse(graph, urls, n_procs) -> dict[url, list]`: parse the outlinks of many pages at
once (parsing is the CPU-bound step), across processes with `ProcessPoolExecutor`; the worker is
`crawler_workers.parse_page` (normalize + dedupe, same rules as Part 1).

**Discuss:** pages are independent (embarrassingly parallel); GIL (parsing is CPU-bound -> processes);
pickling cost of shipping page contents; and that a real crawler overlaps network I/O (threads/async)
with CPU parsing (processes) — a hybrid.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import crawler_workers


def parallel_fetch_parse(graph, urls, n_procs) -> dict:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    g = {"a": ["b", "b", "a", "c"], "b": ["c"], "c": []}
    res = parallel_fetch_parse(g, ["a", "b", "c"], 2)
    assert res["a"] == ["b", "c"] and res["b"] == ["c"] and res["c"] == []
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Politeness: per-host rate limits, robots.txt, dedupe by canonical URL.
- Hybrid concurrency: async/thread fetch (I/O-bound) feeding a process pool for parse (CPU-bound).
- Distributed frontier: sharded queues, bloom-filter "seen" set, restartability.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
from collections import deque
import queue
import threading
from concurrent.futures import ProcessPoolExecutor
import crawler_workers


def ref_get_links(graph, url):
    seen, out = set(), []
    for link in graph.get(url, []):
        if link == url or link in seen:
            continue
        seen.add(link)
        out.append(link)
    return out


def ref_crawl_bfs(graph, seed, max_depth):
    visited, order = {seed}, [seed]
    q = deque([(seed, 0)])
    while q:
        u, d = q.popleft()
        if d >= max_depth:
            continue
        for link in ref_get_links(graph, u):
            if link not in visited:
                visited.add(link)
                order.append(link)
                q.append((link, d + 1))
    return order


def ref_concurrent_crawl(graph, seed, num_workers):
    frontier = queue.Queue()
    visited = {seed}
    lock = threading.Lock()
    frontier.put(seed)

    def worker():
        while True:
            u = frontier.get()
            if u is None:                              # shutdown sentinel
                frontier.task_done()
                return
            for link in ref_get_links(graph, u):
                with lock:                             # guard the check-then-add
                    new = link not in visited
                    if new:
                        visited.add(link)
                if new:
                    frontier.put(link)
            frontier.task_done()

    threads = [threading.Thread(target=worker) for _ in range(num_workers)]
    for t in threads:
        t.start()
    frontier.join()                                    # all enqueued URLs processed
    for _ in threads:
        frontier.put(None)
    for t in threads:
        t.join()
    return visited


def ref_robust_crawl(graph, seed, max_depth):
    if seed not in graph:
        return set(), 0
    visited, dead = {seed}, set()
    q = deque([(seed, 0)])
    while q:
        u, d = q.popleft()
        if d >= max_depth:
            continue
        for link in ref_get_links(graph, u):
            if link not in graph:                      # dead link
                dead.add(link)
                continue
            if link not in visited:
                visited.add(link)
                q.append((link, d + 1))
    return visited, len(dead)


def ref_parallel_fetch_parse(graph, urls, n_procs):
    items = [(u, graph.get(u, [])) for u in urls]
    with ProcessPoolExecutor(max_workers=n_procs) as ex:
        return dict(ex.map(crawler_workers.parse_page, items))


assert ref_get_links({"a": ["a", "b", "b"]}, "a") == ["b"]
_g = {"a": ["b"], "b": ["c"], "c": []}
assert ref_crawl_bfs(_g, "a", 9) == ["a", "b", "c"]
assert ref_concurrent_crawl(_g, "a", 4) == {"a", "b", "c"}
assert ref_robust_crawl({"a": ["b", "zz"], "b": []}, "a", 9) == ({"a", "b"}, 1)
assert ref_parallel_fetch_parse({"a": ["b", "b"]}, ["a"], 2) == {"a": ["b"]}
print("reference solutions OK")